# QEvict completion ablation for Reviewer 2 (Colab, A100)

> *How sensitive is QEvict's model quality to the accuracy of the calibrated window-level
> completion term for windows that are not selected for token-level attention?*

**Model:** Llama-3.1-8B-Instruct. **Operating point:** `utils.config.OPERATING_POINT`: budget 0.20, W = 8, `quant_ratio` 0.70
in `bytes` mode, read ratio ρ = 0.25, int2, flash backend, `bidir`/`dequant`, greedy.

| Experiment | Question | Where |
|---|---|---|
| **1** Matched-state accuracy | How close is the completion mass to the true unread-window mass, and how much output error does each variant cause **on the same decode state**? | collector, `--completion-rungs` |
| **2** Is completion needed? | `cal` vs `off` (and the ρ = 1.0 ceiling) | LongBench |
| **3** Is calibration needed? | `cal` vs `uncal` (`exp(ℓ̂ − log Z_A)`) | LongBench |
| **4** How sensitive is quality? | common scale `c`; total-preserving noise `σ`; coupled scores | LongBench |

`cal` is the shipped method, the control in every experiment. Two floors are measured on 10 examples per task:
* `cal_repeat`, a second run of the same configuration, gives the run-to-run noise;
* `cal_ep` sends the shipped term through the evaluation epilogue at identity, which isolates the epilogue's own numerics.

### The knob (`utils/completion_spec.py`)
* `cache.quant_completion`: `null` runs the shipped kernel path. Any other value (`calibrated`, `off`, `uncalibrated`) runs the
  evaluation epilogue (`modules/windowed_cache/completion.py`). The epilogue recovers `ō`, the fill and `log Z_A` exactly from
  the kernel's own output and applies the variant **after selection and after `δ`**. The kernel source is unchanged.
* `quant_completion_scale`: `c`. `quant_completion_sigma` (in nats) with `quant_completion_seed`: noise that preserves the total,
  drawn per layer × query head × window id and fixed for the window's life. `quant_completion_scores`: `output` leaves the
  eviction scores on the shipped fill; `coupled` moves them too.
* Use `off`, not `none`: the override parser turns `none` into null.

### Why a number can be trusted
Each LongBench sidecar must carry all of the following, or the result is moved to `refused/` and is not scored:
* `read_gate.verdict == "gated"`;
* the realised read fraction for the requested ρ;
* the operating point;
* the requested `num_samples`;
* `quant_completion` equal to the arm's spec;
* runtime proof: `read_gate.completion_applied == read_gate.gated` on variant arms, and `0` on native arms.

One Drive folder holds one commit. **Run every arm on the same commit.** The notebook refuses to start if the knob is missing.

### Run size: an ablation, not a benchmark
| `RUN` | What | One A100 (arithmetic, not measured) |
|---|---|---|
| `"core"` (default) | Exp 1 on 4 tasks × 4 prompts × 128 steps (~0.5M head-steps per task); LongBench at 40 examples/task for `cal`, `off`, `uncal`, `scale2`, `noise1x`; floors at 10/task | **~2–2.5 h** |
| `"extended"` | adds the ρ = 1.0 ceiling, scales 0.5/0.8/1.25/4, more noise seeds and levels, the ρ = 0.10 pair, coupled scores, GSM8K (250) | about +5 h at 40/task |

**The precision comes from Experiment 1.** It scores every variant on the same decode states, at about half a million
head-steps per task: all five scales, all four noise levels and both oracles. That is cheap, and it gives a tight
sensitivity curve. The LongBench arms confirm the key points end to end. At 40 examples per task they detect only
large changes; the power check prints the exact smallest detectable change. `NUM_SAMPLES` scales the time linearly.

The estimates assume ~55 ms/token at B=1 plus prefill, +20% for epilogue arms, and one model load per arm. After the
first arm, the status cell re-projects the rest from the **measured** examples per second.

### How to use
1. Runtime → **A100** (High-RAM helps Experiment 1). Colab secrets: `HF_TOKEN` (Llama-3.1 access), and `GITHUB_TOKEN` if the repo is private.
2. Run top to bottom. Every finished arm is kept on Drive: after a disconnect, run everything again.
3. For more points, set `RUN = "extended"` and run again. Finished arms are skipped.

## 0. Settings

In [ ]:
# ===================== Settings: edit here =====================
BRANCH       = "Clustered_int2_paused_fast"
REPO_URL     = "https://github.com/shaswata965-coder/RecoverKv.git"
PIN_COMMIT   = None      # set to the SHA printed below before any rerun, so every arm runs on the same commit
MODEL_ID     = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_DIR    = "/content/models/Llama-3.1-8B-Instruct"   # the name must contain "instruct": the chat-template gate reads it
DRIVE_ROOT   = "/content/drive/MyDrive/recoverkv_completion_ablation"

RUN          = "core"    # "core" (~2-2.5 h) or "extended" (adds the rest; finished arms are kept)
TASKS        = ["narrativeqa", "musique", "qmsum", "triviaqa"]
NUM_SAMPLES        = 40      # LongBench examples per task, every arm; pairs use the first N. Time scales linearly
NUM_SAMPLES_FLOOR  = 10      # cal_repeat and cal_ep measure floors, not effects
RUN_GSM8K          = False   # extended only
GSM8K_NUM_SAMPLES  = 250     # the full split is 1,319 CoT answers: about 10 h per arm

# Experiment 1: collector on the shipped method, every variant computed on the same states
EXP1_PROMPTS = 4         # x 128 steps x 32 layers x 32 heads = ~0.5M head-steps per task
EXP1_PREFILL = 4096
EXP1_GEN     = 128
EXP1_CAL_VS_KERNEL_MAX = 1e-2   # refuse Exp 1 if the recomputed calibrated output strays this far from the kernel's (median)

BOOTSTRAP_ITERS        = 10_000
RUN_PREFLIGHT_TESTS    = True
FLASH_ATTN_FROM_SOURCE = False   # only if no prebuilt wheel matches (slow: ~1-2 h)
ALLOW_NON_A100         = False

# ===================== Fixed: these define the experiment =====================
OPERATING_POINT = dict(cache_budget=0.20, window_size=8, quant_ratio=0.70,
                       quant_budget_mode="bytes", quant_promotion="bidir",
                       quant_promote_source="dequant")
GATE = 0.25
STAGES = {"core": 1, "extended": 2}
assert RUN in STAGES, RUN

## 1. GPU, Drive, repository

In [ ]:
import subprocess, json, os, sys, shutil, time, math, glob, zipfile, io, re, warnings, difflib
from pathlib import Path

q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                    "--format=csv,noheader,nounits"], capture_output=True, text=True)
if q.returncode != 0:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> A100.")
gpu_name, mem_mib, cc = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
GPU_MEM_GB = float(mem_mib) / 1024
print(f"GPU: {gpu_name}  {GPU_MEM_GB:.0f} GB  sm_{cc.replace('.', '')}")
if cc != "8.0" and not ALLOW_NON_A100:
    raise RuntimeError(f"Expected an A100 (sm_80), got {gpu_name} (sm_{cc}). The kernels were "
                       "compile-checked for sm80. Set ALLOW_NON_A100 = True to override.")
SKIP_OOM = GPU_MEM_GB < 70
print(f"skip_oom = {SKIP_OOM}" + ("  (40 GB card: an OOM example scores 0 THUDM-style and is dropped "
      "from paired comparisons in every arm)" if SKIP_OOM else ""))

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = Path(DRIVE_ROOT)
for d in ("longbench", "gsm8k", "exp1", "logs", "refused", "results"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)

def _secret(k):
    try:
        return userdata.get(k)
    except Exception:
        return None

REPO = Path("/content/RecoverKv")
gh = _secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://x-access-token:{gh}@") if gh else REPO_URL

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:  # never echo the tokenised URL
        raise RuntimeError("git " + args[0] + " failed:\n" + r.stderr.replace(gh or "\0", "***"))
    return r.stdout.strip()

if not REPO.exists():
    _git("clone", "--branch", BRANCH, url, str(REPO))
else:
    _git("fetch", "origin", BRANCH, cwd=REPO)
    _git("checkout", "-B", BRANCH, f"origin/{BRANCH}", cwd=REPO)
if PIN_COMMIT:
    _git("checkout", PIN_COMMIT, cwd=REPO)
SHA = _git("rev-parse", "HEAD", cwd=REPO)
print("commit", SHA, "  <- set PIN_COMMIT to this before stages B and C")

# Quality rows do not compare across commits, so one Drive root holds one commit.
manifest = ROOT / "manifest.json"
if manifest.exists():
    m = json.loads(manifest.read_text())
    if m["sha"] != SHA:
        raise RuntimeError(f"{ROOT} holds results from commit {m['sha'][:10]}, and this clone is "
                           f"{SHA[:10]}. Set PIN_COMMIT = '{m['sha'][:10]}' to resume, or point "
                           "DRIVE_ROOT at a new folder.")
else:
    manifest.write_text(json.dumps({"sha": SHA, "branch": BRANCH, "model": MODEL_ID,
                                    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
                                   indent=2))

## 2. Install the pinned stack

* torch 2.7.1 gives Triton 3.3.1, the version the kernels were compile-checked on.
* transformers is pinned to 4.47.1: the windowed cache refuses anything newer.
* `datasets` stays below 3, as `environment.yml` pins it.
* flash-attn comes from a prebuilt wheel. These were checked to exist for torch 2.7, both C++ ABIs, on cp311 and cp312.

The notebook kernel never imports torch: every job runs as a subprocess, so no runtime restart is needed.

In [ ]:
def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

pip("torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
    "--index-url", "https://download.pytorch.org/whl/cu126")
pip("transformers==4.47.1", "accelerate>=0.25,<2", "datasets>=2.16,<3", "huggingface_hub>=0.24,<1.0",
    "sentencepiece", "protobuf", "einops>=0.7", "scipy", "pyyaml", "tqdm", "pandas", "matplotlib",
    "rouge", "fuzzywuzzy", "python-Levenshtein", "jieba", "pytest")

tv, abi, py = subprocess.check_output([sys.executable, "-c",
    "import torch, sys; print(torch.__version__.split('+')[0]); "
    "print(int(torch._C._GLIBCXX_USE_CXX11_ABI)); "
    "print(f'cp{sys.version_info.major}{sys.version_info.minor}')"], text=True).split()
tmm = ".".join(tv.split(".")[:2])
import urllib.request
def _exists(u):
    try:
        req = urllib.request.Request(u, headers={"Range": "bytes=0-0"})
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status in (200, 206)
    except Exception:
        return False
wheel = None
for v in ("2.8.3", "2.8.2", "2.7.4.post1"):
    u = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v{v}/"
         f"flash_attn-{v}+cu12torch{tmm}cxx11abi{'TRUE' if abi == '1' else 'FALSE'}-{py}-{py}-linux_x86_64.whl")
    if _exists(u):
        wheel = u
        break
if wheel:
    print("flash-attn wheel:", wheel.rsplit("/", 1)[1])
    pip(wheel)
elif FLASH_ATTN_FROM_SOURCE:
    subprocess.run([sys.executable, "-m", "pip", "install", "flash-attn==2.8.3", "--no-build-isolation"],
                   check=True, env=dict(os.environ, MAX_JOBS="8"))
else:
    raise RuntimeError(f"No prebuilt flash-attn wheel for torch {tv} / {py} / cxx11abi={abi}. "
                       "Set FLASH_ATTN_FROM_SOURCE = True (about 1-2 h) and rerun this cell.")

check = r"""
import torch, triton, transformers, flash_attn, datasets
print("torch", torch.__version__, "| triton", triton.__version__, "| transformers", transformers.__version__,
      "| flash_attn", flash_attn.__version__, "| datasets", datasets.__version__)
assert torch.cuda.is_available()
from flash_attn import flash_attn_func
x = torch.randn(1, 256, 8, 128, device="cuda", dtype=torch.float16)
flash_attn_func(x, x, x, causal=True); torch.cuda.synchronize()
assert triton.__version__.startswith("3.3"), "the kernels were compile-checked on Triton 3.3.1"
from utils.cache_factory import assert_transformers_version_supported
assert_transformers_version_supported()
print("environment ok")
"""
subprocess.run([sys.executable, "-c", check], cwd=REPO, check=True)

## 3. Model and data

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
HF_TOKEN = _secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add a Colab secret HF_TOKEN with access to meta-llama/Llama-3.1-8B-Instruct.")
assert "instruct" in Path(MODEL_DIR).name.lower(), "MODEL_DIR's name must contain 'instruct'"
snapshot_download(MODEL_ID, local_dir=MODEL_DIR, token=HF_TOKEN,
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

LB_DIR = REPO / "data" / "longbench"
LB_DIR.mkdir(parents=True, exist_ok=True)
need = [t for t in TASKS if not (LB_DIR / f"{t}.jsonl").exists()]
if need:
    zp = hf_hub_download("THUDM/LongBench", "data.zip", repo_type="dataset", token=HF_TOKEN)
    with zipfile.ZipFile(zp) as z:                      # members are data/<task>.jsonl
        names = z.namelist()
        for t in need:
            member = next(n for n in names if n == f"{t}.jsonl" or n.endswith(f"/{t}.jsonl"))
            with z.open(member) as src, open(LB_DIR / f"{t}.jsonl", "wb") as dst:
                shutil.copyfileobj(src, dst)
for t in TASKS:
    ids = [json.loads(l)["_id"] for l in open(LB_DIR / f"{t}.jsonl")]
    assert len(ids) == len(set(ids)), f"{t}: duplicate _id, paired comparisons would be wrong"
    print(f"{t:12s} {len(ids)} examples, unique _id")

ENV = dict(os.environ, PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8",
           TOKENIZERS_PARALLELISM="false", LONGBENCH_LOCAL_DIR=str(LB_DIR),
           HF_TOKEN=HF_TOKEN, PYTHONUNBUFFERED="1")

if RUN_GSM8K and RUN == "extended" and not (REPO / "data/gsm8k_cot").exists():
    subprocess.run([sys.executable, "-m", "data.gsm8k_loader", "--out", "data/gsm8k_cot"],
                   cwd=REPO, env=ENV, check=True)

## 4. Readiness check (refuses to start if anything is missing)

1. The gate is wired: `fused_gate` has non-test callers.
2. Every config is at the operating point.
3. The completion knob exists on `CacheConfig`.
4. The collector has `--completion-rungs`.
5. Preflight tests pass on this GPU:
   * the completion-ablation suite, i.e. the epilogue against the CPU oracle, the plumbing and the guard;
   * the gate/fill suites;
   * the sm80 compile tests.

In [ ]:
probe = r"""
import json, dataclasses, subprocess, pathlib
from utils.config import CacheConfig
fields = {f.name for f in dataclasses.fields(CacheConfig)}
callers = subprocess.run("grep -rn 'fused_gate' --include='*.py' . | grep -v '^./tests/'",
                         shell=True, capture_output=True, text=True).stdout.splitlines()
src = pathlib.Path("modules/evaluation/observation_collector.py").read_text()
knobs = ("quant_completion", "quant_completion_scale", "quant_completion_sigma",
         "quant_completion_seed", "quant_completion_scores")
print(json.dumps({"knobs": {k: k in fields for k in knobs},
                  "gate_callers": [c for c in callers if "def fused_gate" not in c],
                  "exp1b_rungs": "--completion-rungs" in src}))
"""
info = json.loads(subprocess.check_output([sys.executable, "-c", probe], cwd=REPO, env=ENV, text=True))
op = subprocess.run([sys.executable, "scripts/check_operating_point.py"], cwd=REPO, env=ENV,
                    capture_output=True, text=True)
READY = {"gate_wired": bool(info["gate_callers"]), "operating_point": op.returncode == 0,
         "knob": all(info["knobs"].values()), "exp1b": info["exp1b_rungs"]}
print(f"commit {SHA[:10]}: " + ", ".join(f"{k}={v}" for k, v in READY.items()))
if not all(READY.values()):
    print(op.stdout[-2000:])
    raise RuntimeError("This commit cannot run the experiment: " +
                       ", ".join(k for k, v in READY.items() if not v) +
                       ". Nothing was run: control arms from this commit could not be combined with "
                       "variant arms from a later one.")
if RUN_PREFLIGHT_TESTS:
    t = subprocess.run([sys.executable, "-m", "pytest", "-q", "-x", "-p", "no:cacheprovider",
                        "tests/test_completion_ablation.py", "tests/test_gate_fill_calibration.py",
                        "tests/test_gate_union_is_per_head.py", "tests/test_kernel_compiles.py"],
                       cwd=REPO, env=dict(ENV, ATEN_CPU_CAPABILITY="default"),  # scalar CPU codegen for the
                       capture_output=True, text=True)                           # CPU-side cache tests only
    print(t.stdout[-1200:])
    if t.returncode != 0:
        raise RuntimeError("Preflight tests failed on this GPU. See the output above.")
print("ready")

## 5. Arms and the job runner

In [ ]:
LOGS, RUNS = ROOT / "logs", Path("/content/runs")

class Blocked(Exception):
    pass

def run(cmd, log_name):
    cmd = [str(c) for c in cmd]
    log = LOGS / f"{log_name}.log"
    print("$", " ".join(cmd))
    t0, tail = time.time(), []
    keep = ("examples done", "read gate", "OOM", "Error", "error:", "NOT-GATED", "Traceback", "score=")
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            fh.write(line)
            tail = (tail + [line])[-60:]
            if any(k in line for k in keep):
                print("   ", line.rstrip()[:200])
        rc = p.wait()
    if rc != 0:
        print("".join(tail))
        raise RuntimeError(f"{log_name} failed (rc={rc}). Full log: {log}")
    print(f"    finished in {(time.time() - t0) / 60:.1f} min")

NATIVE = {"mode": "native", "scale": 1.0, "sigma": 0.0, "seed": 0, "scores": "output"}
def C(**kw):
    return {"mode": "calibrated", "scale": 1.0, "sigma": 0.0, "seed": 0, "scores": "output", **kw}

ARMS = {}
def arm(name, stage, exp, gate=GATE, comp=NATIVE, sigma_mult=None, floor=False):
    ARMS[name] = dict(stage=stage, exp=exp, gate=gate, comp=comp, sigma_mult=sigma_mult, floor=floor)

# core: one point per question end to end. Experiment 1 covers every level on matched states.
arm("cal",          "core", "control")
arm("off",          "core", "2", comp=C(mode="off"))
arm("uncal",        "core", "3", comp=C(mode="uncalibrated"))
arm("scale2",       "core", "4A", comp=C(scale=2.0))           # over-crediting: the direction section 11 found harmful
arm("noise1x_s0",   "core", "4B", comp=C(seed=0), sigma_mult=1)
arm("cal_repeat",   "core", "floor", floor=True)               # run-to-run noise
arm("cal_ep",       "core", "floor", comp=C(), floor=True)     # the epilogue at identity
# extended
arm("ceiling_g1.0", "extended", "2", gate=1.0)                 # nothing unread
for c in (0.5, 0.8, 1.25, 4.0):
    arm(f"scale{c:g}", "extended", "4A", comp=C(scale=c))
for s in (1, 2):
    arm(f"noise1x_s{s}", "extended", "4B", comp=C(seed=s), sigma_mult=1)
for mult in (2, 4):
    arm(f"noise{mult}x_s0", "extended", "4B", comp=C(seed=0), sigma_mult=mult)
arm("stress_g0.10_cal", "extended", "2", gate=0.10)
arm("stress_g0.10_off", "extended", "2", gate=0.10, comp=C(mode="off"))
for c in (0.5, 2.0):
    arm(f"coupled_scale{c:g}", "extended", "4C", comp=C(scale=c, scores="coupled"))

SIGMA_FILE = ROOT / "exp1" / "sigma_hat.json"

def n_for(a):
    return NUM_SAMPLES_FLOOR if a["floor"] else NUM_SAMPLES

def resolved_comp(a):
    c = dict(a["comp"])
    if a["sigma_mult"]:
        if not SIGMA_FILE.exists():
            raise Blocked("noise arms need sigma_hat_U from Experiment 1. Run section 6 first")
        c["sigma"] = a["sigma_mult"] * json.loads(SIGMA_FILE.read_text())["sigma_hat_U"]
    return c

def overrides(a, comp):
    ov = [f"model.name={MODEL_DIR}", f"cache.quant_gate_ratio={a['gate']}"]
    if comp["mode"] != "native":
        ov += [f"cache.quant_completion={comp['mode']}", f"cache.quant_completion_scale={comp['scale']}",
               f"cache.quant_completion_sigma={comp['sigma']}", f"cache.quant_completion_seed={comp['seed']}",
               f"cache.quant_completion_scores={comp['scores']}"]
    return ov

def _same(g, v):
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return g is not None and abs(float(g) - float(v)) <= 1e-9 * max(1.0, abs(float(v)))
    return g == v

def verify(meta, a, comp, n=None):
    """Everything a sidecar must say for its number to belong to this arm."""
    errs = []
    rg = meta.get("read_gate") or {}
    if rg.get("verdict") != "gated":
        errs.append(f"read_gate verdict {rg.get('verdict')!r}, need 'gated'")
    rf = rg.get("read_fraction")
    if rf is None or not (a["gate"] - 1e-6 <= rf <= min(1.0, a["gate"] + 0.15) + 1e-6):
        errs.append(f"realised read fraction {rf} for requested ratio {a['gate']}")
    for k, v in OPERATING_POINT.items():
        if k in meta and not _same(meta[k], v):
            errs.append(f"{k} = {meta[k]!r}, expected {v!r}")
    got = meta.get("quant_completion")
    if not isinstance(got, dict):
        errs.append("no quant_completion record in the sidecar")
    else:
        for k, v in comp.items():
            if not _same(got.get(k), v):
                errs.append(f"quant_completion.{k} = {got.get(k)!r}, expected {v!r}")
    applied, gated = rg.get("completion_applied"), rg.get("gated")
    want = 0 if comp["mode"] == "native" else gated
    if applied is None or applied != want or (comp["mode"] != "native" and not gated):
        errs.append(f"completion_applied = {applied}, gated = {gated}: the epilogue must run on every "
                    "gated layer of a variant arm and on none of a native one")
    if n is not None and "num_samples_requested" in meta and meta["num_samples_requested"] != n:
        errs.append(f"num_samples_requested = {meta['num_samples_requested']!r}, expected {n!r}")
    return errs

def _refuse(files, tag, errs):
    dst = ROOT / "refused" / tag
    dst.mkdir(parents=True, exist_ok=True)
    for f in files:
        if Path(f).exists():
            shutil.move(str(f), dst / Path(f).name)
    raise RuntimeError(f"REFUSED {tag}:\n  " + "\n  ".join(errs) + f"\n  files moved to {dst}")

def lb_arm(name):
    """Every pending task of one arm in ONE process (one model load), then each task's
    sidecar verified and copied to Drive. A task that finished before a crash is kept."""
    a = ARMS[name]
    comp, n = resolved_comp(a), n_for(a)
    dst = ROOT / "longbench" / name
    dst.mkdir(parents=True, exist_ok=True)
    todo = []
    for t in TASKS:
        meta_f, pred_f = dst / f"{t}.meta.json", dst / f"{t}.jsonl"
        if meta_f.exists() and pred_f.exists():     # the sidecar is written last: it is the done-marker
            errs = verify(json.loads(meta_f.read_text()), a, comp, n)
            if errs:
                raise RuntimeError(f"STALE {name}/{t} on Drive does not match this arm:\n  "
                                   + "\n  ".join(errs) + "\n  Move it away to rerun.")
        else:
            todo.append(t)
    if not todo:
        return "done"
    local = RUNS / "longbench" / name
    shutil.rmtree(local, ignore_errors=True)    # the runner's own resume trusts a partial jsonl
    local.mkdir(parents=True)
    cmd = [sys.executable, "main.py", "--config", "configs/longbench_ours_flash_attn.yaml", "--override",
           f"longbench.datasets=[{','.join(todo)}]", f"longbench.output_dir={local}",
           f"longbench.num_samples={n}", "longbench.resume=false",
           f"longbench.skip_oom={str(SKIP_OOM).lower()}", *overrides(a, comp)]
    failure = None
    try:
        run(cmd, f"lb_{name}")
    except RuntimeError as e:
        failure = e
    for t in todo:
        mf, pf = local / f"{t}.meta.json", local / f"{t}.jsonl"
        if not (mf.exists() and pf.exists()):
            continue
        errs = verify(json.loads(mf.read_text()), a, comp, n)
        if errs:
            _refuse([pf, mf], f"{name}_{t}", errs)
        shutil.copy(pf, dst / f"{t}.jsonl")
        shutil.copy(mf, dst / f"{t}.meta.json")
    if failure is not None:
        raise failure
    return "ran"

GSM8K_ARMS = ["cal", "off", "uncal", "scale0.5", "scale2"]

def gsm8k_job(name):
    a = ARMS[name]
    comp = resolved_comp(a)
    dst = ROOT / "gsm8k" / name
    if (dst / "meta.json").exists() and (dst / "predictions.jsonl").exists():
        errs = verify(json.loads((dst / "meta.json").read_text()), a, comp, GSM8K_NUM_SAMPLES)
        if errs:
            raise RuntimeError(f"STALE gsm8k/{name}:\n  " + "\n  ".join(errs))
        return "done"
    local = RUNS / "gsm8k" / name
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, "main.py", "--config", "configs/gsm8k_budget20.yaml", "--override",
         f"gsm8k.output_dir={local}", f"gsm8k.num_samples={GSM8K_NUM_SAMPLES}",
         f"gsm8k.skip_oom={str(SKIP_OOM).lower()}", *overrides(a, comp)], f"gsm8k_{name}")
    meta = json.loads((local / "meta.json").read_text())
    errs = verify(meta, a, comp, GSM8K_NUM_SAMPLES)
    if errs:
        _refuse([local / "predictions.jsonl", local / "meta.json"], f"gsm8k_{name}", errs)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(local / "predictions.jsonl", dst / "predictions.jsonl")
    shutil.copy(local / "meta.json", dst / "meta.json")
    return "ran"

def status_table(stage_max):
    names = [n for n, a in ARMS.items() if STAGES[a["stage"]] <= stage_max]
    loads = 0
    rows, remaining_s = [], 0.0
    eps = {}
    for t in TASKS:   # measured examples/second per task, from any finished native arm
        for n in ("cal", "ceiling_g1.0", "off"):
            f = ROOT / "longbench" / n / f"{t}.meta.json"
            if f.exists():
                eps[t] = json.loads(f.read_text()).get("examples_per_second")
                break
    print(f"\n{'arm':20s}{'set':>9s}" + "".join(f"{t:>13s}" for t in TASKS))
    for n in names:
        cells = []
        for t in TASKS:
            done = (ROOT / "longbench" / n / f"{t}.meta.json").exists()
            cells.append("done" if done else "-")
            if not done and eps.get(t):
                remaining_s += n_for(ARMS[n]) / eps[t] * (1.0 if ARMS[n]["comp"]["mode"] == "native" else 1.2)
        print(f"{n:20s}{ARMS[n]['stage']:>9s}" + "".join(f"{c:>13s}" for c in cells))
        loads += "-" in cells
    if eps:
        remaining_s += 90 * loads                  # one model load + compile warm-up per pending arm
        print(f"\nprojected remaining for RUN = {RUN!r}: {remaining_s / 3600:.1f} h "
              "(measured ex/s of finished native jobs; +20% for epilogue arms; ~90 s load per arm)")

def run_grid(stage_max):
    for n, a in ARMS.items():
        if STAGES[a["stage"]] > stage_max:
            continue
        try:
            if lb_arm(n) == "ran":
                status_table(stage_max)            # progress, and the re-projected remaining time
        except Blocked as e:
            print(f"BLOCKED {n}: {e}")
    status_table(stage_max)

## 6. Experiment 1: every variant on the same decode states

One collector run per task on the shipped method, with `--completion-rungs`. At every gated (step, layer, query head)
the collector holds the dequantized int2 tier and the full-KV shadow, so it computes **on the same state**:
* the exact int2 mass of every window over `Z_A`, `m*(u)`;
* the calibrated and uncalibrated masses;
* the output of each variant: `off`, `uncal`, `cal`, five scales, four noise levels, the **mass oracle** (`m*` with the
  stored centroid), the **value oracle** and the **full oracle** (= attention over every held window).

Output errors are against `o_held`, which removes eviction and int2 error and isolates the completion term.

| Statistic | Meaning |
|---|---|
| `unread` | share of held attention on unread int2 windows: what the completion stands in for |
| `E_w`, `E_unw` | `|log m̃_cal − log m*|` over unread windows, mass-weighted and unweighted (the reviewer's `E_logm`; the unweighted form is dominated by near-empty windows) |
| `bias_w` | mean signed log error on U. `δ` is fitted on R, so the R mean is 0 by construction; this is the selection bias |
| `tot_err` | `log Σ_U m̃_cal − log Σ_U m*`, the error on the total unread mass. `c_hat = exp(median)` is marked on 4A |
| `sd_U` | residual spread on U. Its median is `σ̂_U`, the unit of the noise arms |
| `E_w_uncal`, `uncal_factor` | the same for `uncal`; `exp(uncal_shift)` is the per-step common factor `uncal` applies to `cal` |
| rungs | rel. L2 output error vs `o_held`, normalised by `|o_full|` like the collector's ladder |

Sanity gates that refuse the data:
* `full_oracle` must reproduce the collector's own `o_held` (≤ 1e-3);
* `cal` must reproduce the kernel's real output: median `cal_vs_actual` ≤ `EXP1_CAL_VS_KERNEL_MAX`. fp16 storage and the kernel's base-2 softmax account for roughly 1e-3.

Statistics are taken per query head first, then aggregated over heads. Windows below 1e-6 mass are left out of the unweighted
statistics. **Caveat:** prompts are the first `EXP1_PREFILL` tokens of each task's documents plus a greedy continuation, not
the templated task prompt. Collector zips (~0.5–1 GB per prompt) stay on local disk; only the analysed results go to Drive.

In [ ]:
import numpy as np, pandas as pd

FLOOR = 1e-6
RUNGS = None   # read from the zip's meta

def iter_zip(path, keys):
    with zipfile.ZipFile(path) as zf:
        meta = json.loads(zf.read("meta.json"))
        for s in meta["samples"]:
            with np.load(io.BytesIO(zf.read(s["file"])), allow_pickle=False) as z:
                yield {k: z[k] for k in keys}

def _nanmean(x, w=None, axis=-1):
    with np.errstate(invalid="ignore", divide="ignore"):
        if w is None:
            return np.nanmean(x, axis=axis) if np.isfinite(x).any() else np.nan
        return np.nansum(x * w, axis=axis) / np.nansum(w, axis=axis)

def _head_step(tier_tl, go_tl, lms, err, shift, cerr, cva, rep):
    """One (step, layer): a row of statistics per query head (NaN where undefined)."""
    q = (tier_tl == 1)[None, :]
    op = np.repeat(go_tl, rep, axis=0)
    R, U = q & op, q & ~op
    lms = lms.astype(np.float64); e = err.astype(np.float64)
    fin = np.isfinite(lms) & np.isfinite(e)
    m = np.where(fin, np.exp(np.where(fin, lms, -np.inf)), 0.0)
    Uw, Uf = U & fin, U & fin & (m >= FLOOR)
    nU = Uf.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        wU = np.where(Uw, m, 0.0); sw = wU.sum(1)
        e_unc = e + shift[:, None]
        out = {
            "unread": sw / (1.0 + sw),
            "E_w": (wU * np.abs(np.where(Uw, e, 0))).sum(1) / sw,
            "E_unw": np.where(nU > 0, (np.abs(np.where(Uf, e, 0))).sum(1) / nU, np.nan),
            "bias_w": (wU * np.where(Uw, e, 0)).sum(1) / sw,
            "bias_unw": np.where(nU > 0, np.where(Uf, e, 0).sum(1) / nU, np.nan),
            "tot_err": np.log((wU * np.exp(np.where(Uw, e, 0))).sum(1)) - np.log(sw),
            "sd_U": np.where(nU >= 4, np.nanstd(np.where(Uf, e, np.nan), axis=1, ddof=1), np.nan),
            "E_w_uncal": (wU * np.abs(np.where(Uw, e_unc, 0))).sum(1) / sw,
            "uncal_factor": np.exp(shift),
            "cal_vs_actual": cva.astype(np.float64),
        }
    valid = (R.sum(1) > 0) & (Uw.sum(1) > 0)
    for k in out:
        out[k] = np.where(valid & np.isfinite(out[k]), out[k], np.nan)
    rung = np.where(valid[:, None], cerr[..., 0].astype(np.float64), np.nan)     # vs o_held
    rung_full = np.where(valid[:, None], cerr[..., 1].astype(np.float64), np.nan)
    return out, rung, rung_full

EXP1_KEYS = ("tier", "gate_open", "gate_fired", "comp_log_mstar", "comp_logerr_cal",
             "comp_uncal_shift", "comp_err", "comp_cal_vs_actual")

def exp1_analyse(zp):
    with zipfile.ZipFile(zp) as zf:
        meta = json.loads(zf.read("meta.json"))
    rungs = meta["completion_rungs"]
    stats, rg, rgf, ids = {}, [], [], []
    for si, s in enumerate(iter_zip(zp, EXP1_KEYS)):
        T, L, H, _ = s["comp_log_mstar"].shape
        rep = H // s["gate_open"].shape[2]
        for t in range(1, T):
            for l in np.flatnonzero(s["gate_fired"][t]):
                o, r, rf = _head_step(s["tier"][t, l], s["gate_open"][t, l], s["comp_log_mstar"][t, l],
                                      s["comp_logerr_cal"][t, l], s["comp_uncal_shift"][t, l],
                                      s["comp_err"][t, l], s["comp_cal_vs_actual"][t, l], rep)
                for k, v in o.items():
                    stats.setdefault(k, []).append(v.astype(np.float32))
                rg.append(r.astype(np.float32)); rgf.append(rf.astype(np.float32))
                ids.append(np.stack([np.full(H, l), np.arange(H)], 1))
    ids = np.concatenate(ids)
    df = pd.DataFrame({k: np.concatenate(v) for k, v in stats.items()})
    df["layer"], df["head"] = ids[:, 0], ids[:, 1]
    R = pd.DataFrame(np.concatenate(rg), columns=rungs)
    Rf = pd.DataFrame(np.concatenate(rgf), columns=rungs)
    R["layer"] = Rf["layer"] = ids[:, 0]
    R["head"] = Rf["head"] = ids[:, 1]
    return meta, df, R, Rf

def per_head(df):
    return df.groupby(["layer", "head"]).median()

def summarise(df):
    ph = per_head(df)
    worst = {k: (ph[k].abs().max() if k in ("bias_w", "bias_unw", "tot_err") else ph[k].max()) for k in ph}
    return pd.DataFrame({"median head": ph.median(), "p90 head": ph.quantile(0.9), "worst head": pd.Series(worst)})

def exp1_job(task):
    res = ROOT / "exp1" / f"{task}_results.json"
    if res.exists():
        return json.loads(res.read_text())
    zp = RUNS / "exp1" / f"{task}_p{EXP1_PREFILL}_n{EXP1_GEN}_k{EXP1_PROMPTS}.zip"
    zp.parent.mkdir(parents=True, exist_ok=True)
    if not zp.exists():
        run(["bash", "scripts/collect_observations.sh", "--model-path", MODEL_DIR, "--dataset", task,
             "--window-size", OPERATING_POINT["window_size"], "--cache-budget", OPERATING_POINT["cache_budget"],
             "--gate-ratio", GATE, "--quant-ratio", OPERATING_POINT["quant_ratio"],
             "--quant-budget-mode", OPERATING_POINT["quant_budget_mode"],
             "--prefill", EXP1_PREFILL, "--gen", EXP1_GEN, "--num-samples", EXP1_PROMPTS,
             "--completion-rungs", "--out", zp], f"exp1_{task}")
    meta, df, R, Rf = exp1_analyse(zp)
    errs = []
    if (meta.get("read_gate") or {}).get("verdict") != "gated":
        errs.append(f"collector read_gate verdict {(meta.get('read_gate') or {}).get('verdict')!r}")
    fo = float(np.nanmax(R["full_oracle"])) if len(R) else float("nan")
    if not fo <= 1e-3:
        errs.append(f"full_oracle differs from the collector's o_held by up to {fo:.2e}: the rung arithmetic is wrong")
    cva = float(np.nanmedian(df["cal_vs_actual"]))
    if not cva <= EXP1_CAL_VS_KERNEL_MAX:
        errs.append(f"the recomputed calibrated output differs from the kernel's real output by {cva:.2e} "
                    f"(median) > {EXP1_CAL_VS_KERNEL_MAX}: the rungs would describe a different computation "
                    "from the one that ran")
    if errs:
        _refuse([zp], f"exp1_{task}", errs)
    s = summarise(df)
    rung_tab = pd.DataFrame({"median head": R.groupby(["layer", "head"]).median().median(),
                             "p90 head": R.groupby(["layer", "head"]).median().quantile(0.9),
                             "vs full-KV, median head": Rf.groupby(["layer", "head"]).median().median()}
                            ).drop(index=["layer", "head"], errors="ignore")
    out = {"task": task, "head_steps": int(len(df)), "summary": s.to_dict(), "rungs": rung_tab.to_dict(),
           "sigma_hat_U": float(np.nanmedian(df["sd_U"])), "tot_err_median": float(np.nanmedian(df["tot_err"])),
           "uncal_factor_median": float(np.nanmedian(df["uncal_factor"])),
           "cal_vs_actual_median": float(np.nanmedian(df["cal_vs_actual"])),
           "cal_vs_actual_p99": float(np.nanpercentile(df["cal_vs_actual"].dropna(), 99)) if df["cal_vs_actual"].notna().any() else None,
           "full_oracle_max": fo, "sha": SHA}
    per_head(df).to_csv(ROOT / "exp1" / f"{task}_per_head.csv")
    R.groupby(["layer", "head"]).median().to_csv(ROOT / "exp1" / f"{task}_rungs_per_head.csv")
    res.write_text(json.dumps(out, indent=1, default=float))
    return out

def exp1_all():
    E1 = {t: exp1_job(t) for t in TASKS}
    for t, r in E1.items():
        print(f"\n=== {t}: {r['head_steps']} head-steps   cal vs kernel output: median {r['cal_vs_actual_median']:.2e}, "
              f"p99 {r['cal_vs_actual_p99']:.2e}   full oracle vs o_held: max {r['full_oracle_max']:.1e}")
        print(pd.DataFrame(r["summary"]).round(4).to_string())
        print(pd.DataFrame(r["rungs"]).round(5).to_string())
    sig = float(np.median([r["sigma_hat_U"] for r in E1.values()]))
    chat = float(np.exp(np.median([r["tot_err_median"] for r in E1.values()])))
    ufac = float(np.median([r["uncal_factor_median"] for r in E1.values()]))
    if SIGMA_FILE.exists():
        old = json.loads(SIGMA_FILE.read_text())
        if abs(old["sigma_hat_U"] - sig) > 1e-9:
            raise RuntimeError(f"sigma_hat_U changed ({old['sigma_hat_U']} -> {sig}); noise arms already on Drive "
                               "were run at the old value. Keep the old Exp 1 results or start a new DRIVE_ROOT.")
    SIGMA_FILE.write_text(json.dumps({"sigma_hat_U": sig, "c_hat": chat, "uncal_factor": ufac, "floor": FLOOR,
                                      "sha": SHA, "per_task": {t: {k: r[k] for k in ("sigma_hat_U", "tot_err_median",
                                      "uncal_factor_median")} for t, r in E1.items()}}, indent=2))
    print(f"\nsigma_hat_U = {sig:.3f} nats (noise-arm unit)   c_hat = {chat:.3f} (measured common-scale error of "
          f"the fill; marked on 4A)   uncal factor = {ufac:.3f} (uncal = cal x this, per step; compare with 4A)")
    for t, r in E1.items():
        m = r["rungs"]["median head"]
        print(f"{t:12s} matched states: completion removes {1 - m['cal'] / m['off']:.1%} of the read-gate output "
              f"error (off {m['off']:.4f} -> cal {m['cal']:.4f}); a perfect mass estimate would reach "
              f"{m['mass_oracle']:.4f}, the stored centroid's floor")
    return E1

E1 = exp1_all()

## 7. Experiments 2–4: the LongBench arms for the current `RUN`

In [ ]:
run_grid(STAGES[RUN])

In [ ]:
if RUN_GSM8K and RUN == "extended":
    for n in GSM8K_ARMS:
        print(n, gsm8k_job(n))

## 8. Scoring, power check, analysis

* Per-example scores use the repo's own LongBench metric functions and first-line rule.
* Task scores follow THUDM: a null (OOM) prediction scores 0.
* Paired deltas use examples that are non-null in both arms, matched by `_id`. Each comes with a 95% bootstrap CI.
* **Behavioural change** is reported against `cal`:
  * the share of examples whose prediction text changed at all;
  * the mean text similarity.

  Read it against the two floors, `cal_repeat` and `cal_ep`.
* **Power check:** the minimum detectable effect (80% power, two-sided 5%) at the current N, from the measured paired spread.
  End to end, the core run only resolves large changes. Experiment 1 carries the precise answer.
* Every sidecar is re-verified, and a file that fails is not scored.

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from modules.evaluation.longbench_scoring import METRIC_FN_REGISTRY, _FIRST_LINE_DATASETS
D2M = json.loads((REPO / "data/longbench_configs/dataset2metric.json").read_text())

def example_scores(jsonl, task):
    fn, out, preds = METRIC_FN_REGISTRY[D2M[task]], {}, {}
    for line in open(jsonl, encoding="utf-8"):
        if not line.strip():
            continue
        ex = json.loads(line)
        pred = ex.get("pred")
        preds[ex["_id"]] = pred
        if pred is None:
            out[ex["_id"]] = None
            continue
        if task in _FIRST_LINE_DATASETS:
            pred = pred.lstrip("\n").split("\n")[0]
        out[ex["_id"]] = 100.0 * max((fn(pred, gt, all_classes=ex.get("all_classes") or [])
                                      for gt in ex.get("answers", [])), default=0.0)
    return out, preds

SCORES, PREDS, SKIPPED = {}, {}, []
for name, a in ARMS.items():
    for t in TASKS:
        d = ROOT / "longbench" / name
        if not (d / f"{t}.meta.json").exists():
            continue
        try:
            comp = resolved_comp(a)
        except Blocked:
            continue
        errs = verify(json.loads((d / f"{t}.meta.json").read_text()), a, comp, n_for(a))
        if errs:
            SKIPPED.append((name, t, errs))
            continue
        SCORES[(name, t)], PREDS[(name, t)] = example_scores(d / f"{t}.jsonl", t)
for s in SKIPPED:
    print("NOT SCORED", s)

rng = np.random.default_rng(0)
def paired(x, y):
    ids = sorted(i for i in set(x) & set(y) if x[i] is not None and y[i] is not None)
    return np.array([x[i] for i in ids]), np.array([y[i] for i in ids]), ids

def boot_dist(d):
    idx = rng.integers(0, len(d), (BOOTSTRAP_ITERS, len(d)))
    return d[idx].mean(1)

arms_present = [n for n in ARMS if any((n, t) in SCORES for t in TASKS)]
tab = pd.DataFrame({t: {n: (np.mean([v if v is not None else 0.0 for v in SCORES[(n, t)].values()])
                            if (n, t) in SCORES else np.nan) for n in arms_present} for t in TASKS})
tab["average"] = tab[TASKS].mean(1)
print(f"Task scores (THUDM convention; {NUM_SAMPLES} examples per task, floors {NUM_SAMPLES_FLOOR}):")
print(tab.round(2).to_string())

rows = []
for n in arms_present:
    if n == "cal" or not all((n, t) in SCORES and ("cal", t) in SCORES for t in TASKS):
        continue
    dists, means, sds, ns, row = [], [], [], [], {"arm": n, "exp": ARMS[n]["exp"]}
    same, sim = [], []
    for t in TASKS:
        x, y, ids = paired(SCORES[(n, t)], SCORES[("cal", t)])
        d = x - y
        dists.append(boot_dist(d)); means.append(d.mean()); sds.append(d.std(ddof=1)); ns.append(len(d))
        lo, hi = np.percentile(dists[-1], [2.5, 97.5])
        row[t] = f"{d.mean():+.2f} [{lo:+.2f},{hi:+.2f}]"
        pa, pb = PREDS[(n, t)], PREDS[("cal", t)]
        same += [pa[i] == pb[i] for i in ids]
        sim += [difflib.SequenceMatcher(None, pa[i], pb[i]).ratio() for i in ids]
    row["avg_delta"] = float(np.mean(means))
    row["avg_lo"], row["avg_hi"] = np.percentile(np.mean(dists, 0), [2.5, 97.5])
    row["pred_changed"] = 1.0 - float(np.mean(same))
    row["pred_similarity"] = float(np.mean(sim))
    se_avg = math.sqrt(sum((s / math.sqrt(k)) ** 2 for s, k in zip(sds, ns))) / len(TASKS)
    row["MDE80_avg"] = 2.80 * se_avg
    row["n_paired"] = int(np.sum(ns))
    rows.append(row)
DELTA = pd.DataFrame(rows).set_index("arm") if rows else pd.DataFrame()
if len(DELTA):
    print("\nPaired Δ vs cal, points [95% CI]. pred_changed / pred_similarity: behavioural change vs cal "
          "(read against cal_repeat and cal_ep). MDE80_avg: the smallest true change of the 4-task average this N "
          "detects with 80% power.")
    print(DELTA[[*TASKS, "avg_delta", "avg_lo", "avg_hi", "pred_changed", "pred_similarity",
                 "MDE80_avg", "n_paired"]].round(3).to_string())
    DELTA.to_csv(ROOT / "results" / "paired_deltas.csv")
    if "off" in DELTA.index:
        mde = DELTA.loc["off", "MDE80_avg"]
        print(f"\nPOWER CHECK: at the current N, a change of the 4-task average smaller than {mde:.2f} points "
              "is not reliably detectable end to end (it scales with 1/sqrt(NUM_SAMPLES)). The fine-grained "
              "sensitivity is Experiment 1's matched-state curve.")
tab.to_csv(ROOT / "results" / "task_scores.csv")

if all(("off", t) in SCORES and ("ceiling_g1.0", t) in SCORES for t in TASKS):
    num, den = [], []
    for t in TASKS:
        ids = sorted(set.intersection(*[{i for i, v in SCORES[(n, t)].items() if v is not None}
                                        for n in ("cal", "off", "ceiling_g1.0")]))
        c = np.array([SCORES[("cal", t)][i] for i in ids])
        o = np.array([SCORES[("off", t)][i] for i in ids])
        g = np.array([SCORES[("ceiling_g1.0", t)][i] for i in ids])
        idx = rng.integers(0, len(ids), (BOOTSTRAP_ITERS, len(ids)))
        num.append((c - o)[idx].mean(1)); den.append((g - o)[idx].mean(1))
    N, D = np.mean(num, 0), np.mean(den, 0)
    if np.percentile(D, 2.5) > 0:
        gc = N / D
        print(f"\nExperiment 2, share of the read-gate gap closed by completion: {np.median(gc):.2f} "
              f"[{np.percentile(gc, 2.5):.2f}, {np.percentile(gc, 97.5):.2f}]")
    else:
        print("\nExperiment 2 gap closed: undefined, because the ceiling does not beat 'off' significantly.")

if RUN_GSM8K and (ROOT / "gsm8k" / "cal" / "meta.json").exists():
    from modules.evaluation.gsm8k_scoring import gsm8k_score
    G = {}
    for n in GSM8K_ARMS:
        d = ROOT / "gsm8k" / n
        if (d / "meta.json").exists() and not verify(json.loads((d / "meta.json").read_text()), ARMS[n],
                                                     resolved_comp(ARMS[n]), GSM8K_NUM_SAMPLES):
            G[n] = {json.loads(l)["id"]: (None if json.loads(l).get("pred") is None else
                    100.0 * gsm8k_score(json.loads(l)["pred"], json.loads(l)["answer"]))
                    for l in open(d / "predictions.jsonl") if l.strip()}
    for n, s in G.items():
        if n != "cal":
            x, y, _ = paired(s, G["cal"])
            lo, hi = np.percentile(boot_dist(x - y), [2.5, 97.5])
            print(f"GSM8K {n:12s} Δ vs cal {np.mean(x - y):+.2f} [{lo:+.2f}, {hi:+.2f}]  (n={len(x)})")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
BLUE, MUTED = "#2a78d6", "#52514e"
SH = json.loads(SIGMA_FILE.read_text()) if SIGMA_FILE.exists() else {}

def _ref(ax, x, label, side="left"):
    ax.axvline(x, color=MUTED, ls="--", lw=1, zorder=1)
    ax.annotate(label, (x, 1), xycoords=("data", "axes fraction"), textcoords="offset points",
                xytext=(-4 if side == "left" else 4, -4), ha="right" if side == "left" else "left",
                va="top", color=MUTED, fontsize=9)

def _err(ax, xs, ys, lo, hi, **kw):
    o = np.argsort(xs)
    ax.errorbar(np.array(xs)[o], np.array(ys)[o], yerr=[np.array(lo)[o], np.array(hi)[o]],
                color=BLUE, lw=2, marker="o", ms=8, capsize=3, zorder=3, **kw)

floor = max([abs(DELTA.loc[n, "avg_delta"]) for n in ("cal_repeat", "cal_ep") if n in DELTA.index] or [0.0])
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
# top: end to end (Experiment 4)
ax = axes[0, 0]; ax.axhline(0, color=MUTED, lw=1, zorder=1); ax.axhspan(-floor, floor, color="#e6e5e1", lw=0)
xs, ys, lo, hi = [0.0], [0.0], [0.0], [0.0]
for c in (0.5, 0.8, 1.25, 2.0, 4.0):
    n = f"scale{c:g}"
    if n in DELTA.index:
        xs.append(np.log2(c)); ys.append(DELTA.loc[n, "avg_delta"])
        lo.append(DELTA.loc[n, "avg_delta"] - DELTA.loc[n, "avg_lo"]); hi.append(DELTA.loc[n, "avg_hi"] - DELTA.loc[n, "avg_delta"])
_err(ax, xs, ys, lo, hi)
if "off" in DELTA.index:
    x0 = min(xs) - 1.2
    ax.errorbar([x0], [DELTA.loc["off", "avg_delta"]],
                yerr=[[DELTA.loc["off", "avg_delta"] - DELTA.loc["off", "avg_lo"]],
                      [DELTA.loc["off", "avg_hi"] - DELTA.loc["off", "avg_delta"]]],
                color=BLUE, marker="s", ms=8, capsize=3, lw=0, elinewidth=2, zorder=3)
    ax.annotate("off (c = 0)", (x0, DELTA.loc["off", "avg_delta"]), textcoords="offset points",
                xytext=(8, 0), va="center", color=MUTED)
if SH:
    _ref(ax, np.log2(SH["c_hat"]), f"measured fill error\nc = {SH['c_hat']:.2f}")
ax.set_xlabel("log2 c  (common scale on unread-window mass)"); ax.set_ylabel("Δ LongBench average vs cal (points)")
ax.set_title("4A, end to end: common-scale mass error", loc="left")

ax = axes[0, 1]; ax.axhline(0, color=MUTED, lw=1, zorder=1); ax.axhspan(-floor, floor, color="#e6e5e1", lw=0)
xs, ys, lo, hi = [0], [0.0], [0.0], [0.0]
for mult in (1, 2, 4):
    v = [DELTA.loc[f"noise{mult}x_s{s}", "avg_delta"] for s in range(3) if f"noise{mult}x_s{s}" in DELTA.index]
    if v:
        xs.append(mult); ys.append(np.mean(v)); lo.append(np.mean(v) - min(v)); hi.append(max(v) - np.mean(v))
_err(ax, xs, ys, lo, hi)
_ref(ax, 1, "measured residual\nspread " + r"$\hat{\sigma}_U$")
ax.set_xlabel(r"$\sigma\,/\,\hat{\sigma}_U$" + "  (bars: min–max over 3 seeds)")
ax.set_title("4B, end to end: redistribution, total preserved", loc="left")

# bottom: the same perturbations on matched states (Experiment 1)
if E1:
    rung = lambda r: float(np.median([E1[t]["rungs"]["median head"][r] for t in TASKS]))
    ax = axes[1, 0]
    xs = [np.log2(c) for c in (0.5, 0.8, 1.0, 1.25, 2.0, 4.0)]
    ys = [rung(r) for r in ("scale0.5", "scale0.8", "cal", "scale1.25", "scale2", "scale4")]
    ax.plot(xs, ys, color=BLUE, lw=2, marker="o", ms=8, zorder=3)
    for r, lab, x, ha in (("off", "off", 0, "left"), ("uncal", "uncal", 1, "right"),
                          ("mass_oracle", "mass oracle", 0, "left")):
        ax.axhline(rung(r), color=MUTED, lw=1, ls=":", zorder=1)
        ax.annotate(lab, (x, rung(r)), xycoords=("axes fraction", "data"), textcoords="offset points",
                    xytext=(4 if ha == "left" else -4, 3), ha=ha, color=MUTED, fontsize=9)
    if SH:
        _ref(ax, np.log2(SH["c_hat"]), f"c = {SH['c_hat']:.2f}")
    ax.set_xlabel("log2 c"); ax.set_ylabel("output error vs o_held (median head)")
    ax.set_title("Exp 1, matched states: common scale", loc="left")
    ax = axes[1, 1]
    sig = [0.25, 0.5, 1.0, 2.0]
    ax.plot([0] + sig, [rung("cal")] + [rung(f"noise{s:g}") for s in sig], color=BLUE, lw=2, marker="o", ms=8, zorder=3)
    if SH:
        _ref(ax, SH["sigma_hat_U"], r"$\hat{\sigma}_U$" + f" = {SH['sigma_hat_U']:.2f}", side="right")
    ax.set_xlabel(r"$\sigma$ (nats)"); ax.set_title("Exp 1, matched states: redistribution", loc="left")
fig.text(0.01, -0.02, "Top: grey band = largest |Δ| of the two floors (cal_repeat, cal_ep). Bottom: relative L2 error "
         "of the attention output vs reading every held window, same decode states for every point.",
         color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(ROOT / "results" / "sensitivity.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Rebuttal table and run status

In [ ]:
def cell(n, t):
    return tab.loc[n, t] if n in tab.index and not np.isnan(tab.loc[n, t]) else None

RUNG_OF = {"cal": "cal", "off": "off", "uncal": "uncal", "scale0.5": "scale0.5", "scale0.8": "scale0.8",
           "scale1.25": "scale1.25", "scale2": "scale2", "scale4": "scale4"}
label = {"cal": "Calibrated completion (QEvict)", "off": "Without completion", "uncal": "Uncalibrated completion",
         "scale0.5": "Perturbed masses, 0.5x", "scale0.8": "Perturbed masses, 0.8x",
         "scale1.25": "Perturbed masses, 1.25x", "scale2": "Perturbed masses, 2x", "scale4": "Perturbed masses, 4x",
         "noise1x_s0": "Redistribution noise, 1x sigma_hat (seed 0)", "ceiling_g1.0": "All windows read (rho = 1.0)",
         "cal_repeat": "Calibrated, repeat (floor)", "cal_ep": "Calibrated via epilogue (floor)"}
def rung_err(n):
    r = RUNG_OF.get(n)
    if not (E1 and r):
        return "—"
    return f"{np.median([E1[t]['rungs']['median head'][r] for t in TASKS]):.4f}"
lines = ["| Configuration | " + " | ".join(TASKS) + " | Average | Δ vs cal [95% CI] | output error, matched states |",
         "|---|" + "---|" * (len(TASKS) + 3)]
for n, lab in label.items():
    vals = [cell(n, t) for t in TASKS]
    if all(v is None for v in vals):
        lines.append(f"| {lab} | " + " | ".join("—" for _ in TASKS) + f" | — | not run | {rung_err(n)} |")
        continue
    avg = np.mean(vals) if all(v is not None for v in vals) else None
    d = (f"{DELTA.loc[n, 'avg_delta']:+.2f} [{DELTA.loc[n, 'avg_lo']:+.2f}, {DELTA.loc[n, 'avg_hi']:+.2f}]"
         if n in DELTA.index else "—")
    lines.append(f"| {lab} | " + " | ".join("—" if v is None else f"{v:.2f}" for v in vals)
                 + f" | {'—' if avg is None else f'{avg:.2f}'} | {d} | {rung_err(n)} |")
if E1:
    lines.append("")
    lines.append("| Mass estimation (Exp 1, median head) | " + " | ".join(TASKS) + " |")
    lines.append("|---|" + "---|" * len(TASKS))
    for k, lab in (("E_w", "E_logm, mass-weighted (cal)"), ("E_unw", "E_logm, unweighted (cal)"),
                   ("E_w_uncal", "E_logm, mass-weighted (uncal)"), ("bias_w", "bias on unread windows (cal)"),
                   ("unread", "share of held attention unread")):
        lines.append(f"| {lab} | " + " | ".join(f"{E1[t]['summary']['median head'][k]:.4f}" for t in TASKS) + " |")
    lines.append("| read-gate output error removed by completion, 1 - err(cal)/err(off) | " + " | ".join(
        f"{1 - E1[t]['rungs']['median head']['cal'] / E1[t]['rungs']['median head']['off']:.1%}" for t in TASKS) + " |")
md_table = "\n".join(lines)
(ROOT / "results" / "rebuttal_table.md").write_text(md_table + f"\n\ncommit {SHA}\n")
print(md_table)
status_table(STAGES[RUN])
print(f"\nAll results: {ROOT / 'results'}   commit {SHA[:10]}")